# `train_grpo.ipynb` — Cell-by-Cell Code
> Unsloth + TRL GRPO training for OpenEnv Urban Planner  
> Designed for Colab A100/L4. Paste each block into a new cell.

---

## Cell 1 — Markdown (title)

In [ ]:
# OpenEnv Urban Planner — GRPO Training
Train **Qwen2.5-7B-Instruct** with GRPO on the Urban Planner environment.
The model learns long-horizon spatial reasoning: zoning, infrastructure,
budget management over a 16x16 city across 24 seasons.

---

## Cell 2 — Install dependencies

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q trl transformers accelerate datasets peft bitsandbytes matplotlib numpy pydantic

---

## Cell 3 — GPU check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

---

## Cell 4 — Clone repo & add to sys.path

In [ ]:
import subprocess, sys, os

# Replace with your HF Space URL or local path
REPO_URL = "https://huggingface.co/spaces/YOUR_HF_USERNAME/openenv-urban-planner"
REPO_DIR = "repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# We call the simulation directly (no HTTP server needed in Colab)
sys.path.insert(0, os.path.abspath(REPO_DIR))

from server.city_simulation import CitySimulation, SHAPED_REWARDS
from server.urban_planner_environment import UrbanPlannerEnvironment
from models import EpisodeConfig
print("Imports OK")

---

## Cell 5 — Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto (bfloat16 on Ampere+)
    load_in_4bit   = True,      # QLoRA
)
print("Model loaded:", model.config.name_or_path)

---

## Cell 6 — Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model.print_trainable_parameters()

---

## Cell 7 — System prompt & observation formatter

In [ ]:
SYSTEM_PROMPT = """You are an urban planner for a growing city. You have a budget and tools
to zone land and place infrastructure. Your decisions cascade: roads reduce congestion,
schools prevent population decline, flood barriers protect infrastructure.
Plan for the long term. Think before each tool call.

A planning_log is injected into every observation — use it to reason about cause and
effect across seasons. Respect policy_constraints listed — violations lower your score.

Available tools:
  get_city_state(region)
  get_district_report(district_id)
  place_zone(x, y, zone_type, density)
  place_infrastructure(x, y, infra_type)
  allocate_budget(category, amount)
  query_residents(district_id)
  query_traffic_model(origin, destination)
  advance_season()
  get_event_log(last_n)
  get_budget_report()

Respond with reasoning then a JSON tool call:
<tool_call>
{"name": "<tool_name>", "arguments": {<args_dict>}}
</tool_call>"""


def observation_to_text(obs) -> str:
    lines = [
        f"Season: {obs.season} | Budget: ${obs.budget_remaining:,}",
        f"Recent events: {obs.event_log or 'none'}",
    ]
    if obs.policy_constraints:
        lines.append("Policy constraints: " + "; ".join(obs.policy_constraints))
    if obs.planning_log:
        lines.append("Planning log:")
        for e in obs.planning_log[-4:]:
            lines.append(f"  S{e.season}: {e.action_summary} -> {e.consequence[:80]} (dr={e.reward_delta:+.3f})")
    lines.append(f"\nLast tool result:\n{obs.tool_result[:400]}")
    return "\n".join(lines)


def format_prompt(obs_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": obs_text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

---

## Cell 8 — Build training dataset

In [ ]:
import random
from datasets import Dataset

N_EPISODES = 200
SEEDS = random.sample(range(10_000), N_EPISODES)


def make_prompt_record(seed: int) -> dict:
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=10_000))
    return {
        "prompt": format_prompt(observation_to_text(obs)),
        "seed":   seed,
    }


print("Building dataset ...")
train_dataset = Dataset.from_list([make_prompt_record(s) for s in SEEDS])
print(f"Dataset: {len(train_dataset)} rows")
print(train_dataset[0]["prompt"][:400])

---

## Cell 9 — Tool-call parser

In [ ]:
import re, json

VALID_TOOLS = {
    "get_city_state", "get_district_report", "place_zone",
    "place_infrastructure", "allocate_budget", "query_residents",
    "query_traffic_model", "advance_season", "get_event_log", "get_budget_report",
}

_TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)


def parse_tool_call(text: str):
    match = _TOOL_RE.search(text)
    if not match:
        return None
    try:
        payload = json.loads(match.group(1))
        name = payload.get("name", "")
        args = payload.get("arguments", {})
        if name not in VALID_TOOLS:
            return None
        return name, args
    except (json.JSONDecodeError, KeyError):
        return None


# Smoke test
sample = '<tool_call>\n{"name": "place_zone", "arguments": {"x": 3, "y": 4, "zone_type": "residential", "density": 2}}\n</tool_call>'
print(parse_tool_call(sample))

---

## Cell 10 — Reward function

In [ ]:
def reward_fn(completions: list[str], seed: list[int] = None, **kwargs) -> list[float]:
    """
    GRPO reward function.
    - Parses <tool_call> from each completion.
    - Executes it in a fresh env reset from the row's seed.
    - Returns obs.reward (shaped between seasons, rubric at season boundary).
    - Bad parse / invalid tool -> penalty.
    """
    if seed is None:
        seed = [42] * len(completions)

    rewards = []
    for completion, s in zip(completions, seed):
        parsed = parse_tool_call(completion)
        if parsed is None:
            rewards.append(-0.1)   # no valid tool call found
            continue

        tool_name, arguments = parsed
        try:
            env = UrbanPlannerEnvironment()
            env.reset(EpisodeConfig(seed=int(s), starting_budget=10_000))
            obs = env.step({"tool_name": tool_name, "arguments": arguments})
            r = float(obs.reward)
            rewards.append(r if r != 0.0 else 0.001)
        except Exception as exc:
            print(f"[reward_fn] error: {exc}")
            rewards.append(-0.05)

    return rewards


# Smoke test
dummy = '<tool_call>\n{"name": "get_city_state", "arguments": {"region": "all"}}\n</tool_call>'
print("Smoke-test reward:", reward_fn([dummy], seed=[42]))

---

## Cell 11 — GRPO config

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    # Core
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,    # effective batch = 16
    learning_rate               = 1e-5,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    max_grad_norm               = 0.1,
    # GRPO-specific
    num_generations             = 4,    # completions sampled per prompt
    max_new_tokens              = 256,
    temperature                 = 0.9,
    # I/O
    output_dir                  = "./urban_planner_grpo",
    logging_steps               = 10,
    save_steps                  = 100,
    save_total_limit            = 3,
    bf16                        = is_bfloat16_supported(),
    fp16                        = not is_bfloat16_supported(),
    report_to                   = "none",  # swap to "wandb" if desired
    seed                        = 3407,
    dataloader_num_workers      = 0,
)
print("GRPOConfig ready")

---

## Cell 12 — Build trainer and train

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [reward_fn],
    args             = grpo_config,
    train_dataset    = train_dataset,
)

print("Starting training ...")
trainer.train()
print("Training complete")

---

## Cell 13 — Plot training reward curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("assets/plots", exist_ok=True)

log_history = trainer.state.log_history
steps, rews = [], []
for entry in log_history:
    if "reward" in entry:
        steps.append(entry["step"])
        rews.append(entry["reward"])


def smooth(arr, w=10):
    return np.convolve(arr, np.ones(w) / w, mode="valid")


fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, rews, alpha=0.3, color="#6c8ebf", label="raw")
if len(rews) >= 10:
    ax.plot(steps[9:], smooth(rews), color="#6c8ebf", lw=2, label="smoothed (w=10)")
ax.set_xlabel("Step")
ax.set_ylabel("Mean reward")
ax.set_title("Urban Planner GRPO — Training Reward Curve")
ax.legend()
fig.tight_layout()
fig.savefig("assets/plots/reward_curve.png", dpi=150)
plt.show()
print("Saved assets/plots/reward_curve.png")

---

## Cell 14 — Save model

In [ ]:
SAVE_DIR   = "./urban_planner_grpo_final"
HF_REPO_ID = "YOUR_HF_USERNAME/urban-planner-grpo"   # <- edit

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

# Uncomment to push LoRA adapters to HF Hub:
# model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)

---

## Cell 15 — Random agent baseline

In [ ]:
import random as _random

def run_random_agent(seed: int = 999, max_steps: int = 40) -> list[float]:
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=10_000))
    rewards = []
    for _ in range(max_steps):
        if obs.done:
            break
        x, y = _random.randint(0, 15), _random.randint(0, 15)
        action = _random.choice([
            {"tool_name": "get_city_state",      "arguments": {"region": "all"}},
            {"tool_name": "place_zone",          "arguments": {"x": x, "y": y,
                "zone_type": _random.choice(["residential","commercial","green"]),
                "density": _random.randint(1, 2)}},
            {"tool_name": "place_infrastructure","arguments": {"x": x, "y": y,
                "infra_type": _random.choice(["road","school","flood_barrier"])}},
            {"tool_name": "query_residents",     "arguments": {"district_id": _random.randint(0,15)}},
            {"tool_name": "advance_season",      "arguments": {}},
            {"tool_name": "get_budget_report",   "arguments": {}},
        ])
        obs = env.step(action)
        rewards.append(obs.reward)
    return rewards


print("Running random agent baseline ...")
baseline_rewards = run_random_agent(seed=999, max_steps=40)
print(f"Random agent — mean={np.mean(baseline_rewards):.4f}  steps={len(baseline_rewards)}")

---

## Cell 16 — Run trained agent

In [ ]:
FastLanguageModel.for_inference(model)

def run_trained_agent(seed: int = 999, max_steps: int = 40):
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=10_000))
    rewards = []

    for _ in range(max_steps):
        if obs.done:
            break
        prompt_str = format_prompt(observation_to_text(obs))
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens = 256,
                temperature    = 0.6,
                do_sample      = True,
                pad_token_id   = tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = parse_tool_call(completion)
        if parsed:
            obs = env.step({"tool_name": parsed[0], "arguments": parsed[1]})
        else:
            obs = env.step({"tool_name": "get_city_state", "arguments": {"region": "all"}})
        rewards.append(obs.reward)

    return rewards, env


print("Running trained agent ...")
trained_rewards, trained_env = run_trained_agent(seed=999, max_steps=40)
print(f"Trained agent — mean={np.mean(trained_rewards):.4f}  steps={len(trained_rewards)}")

---

## Cell 17 — Comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(baseline_rewards, color="#e07070", lw=1.5, alpha=0.8, label="Random agent")
ax.plot(trained_rewards,  color="#5a9e6f", lw=2,              label="GRPO-trained agent")
ax.axhline(np.mean(baseline_rewards), color="#e07070", ls="--", lw=0.8, alpha=0.5)
ax.axhline(np.mean(trained_rewards),  color="#5a9e6f", ls="--", lw=0.8, alpha=0.5)
ax.set_xlabel("Step")
ax.set_ylabel("Reward")
ax.set_title("Urban Planner — Trained vs. Random Agent (seed=999)")
ax.legend()
fig.tight_layout()
fig.savefig("assets/plots/reward_comparison.png", dpi=150)
plt.show()
print("Saved assets/plots/reward_comparison.png")

---

## Cell 18 — Collapse narratives & ASCII snapshots

In [ ]:
import json, pathlib

COLLAPSE_DIR = pathlib.Path("assets/collapse_cases")
COLLAPSE_DIR.mkdir(parents=True, exist_ok=True)

sim_random  = run_random_agent.__globals__   # just use sim from the trained env
sim_trained = trained_env._sim

# Random agent narrative — rerun to get collapse report
def full_episode_random(seed: int = 1337):
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=10_000))
    for _ in range(40):
        if obs.done: break
        obs = env.step({"tool_name": "advance_season", "arguments": {}})
    sim = env._sim
    report = sim.generate_collapse_report() if sim.is_terminal() and sim.season < 24 else None
    return sim, report

sim_r, cr = full_episode_random(seed=1337)

print("=== Random Agent (seed=1337) ===")
print(f"Survived {sim_r.season} seasons | pop={sim_r._total_population()} | budget=${sim_r.budget:,}")
if cr:
    print(f"Trigger:     {cr.trigger}")
    print(f"Key mistake: {cr.agent_mistake}")
    with open(COLLAPSE_DIR / "random_seed1337.json", "w") as f:
        json.dump(cr.model_dump(), f, indent=2)
    print("Saved collapse report.")

print("\n=== Trained Agent (seed=999) ===")
sim_t = trained_env._sim
print(f"Survived {sim_t.season} seasons | pop={sim_t._total_population()} | budget=${sim_t.budget:,}")
if sim_t.is_terminal() and sim_t.season < 24:
    cr_t = sim_t.generate_collapse_report()
    print(f"Collapsed: {cr_t.trigger}")
else:
    print("No collapse — city survived!")

print("\n=== ASCII snapshots (trained, season 0 and 12) ===")
snaps = sim_t._ascii_snapshots
print(snaps.get(0, "(no S0 snapshot)"))
print(snaps.get(12, "(S12 not reached)"))

---

## Cell 19 — Push adapters to HF Hub

In [ ]:
HF_REPO_ID = "YOUR_HF_USERNAME/urban-planner-grpo"   # <- edit

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")